#**SuFinex Synthetic Data Generator**

This notebook generates realistic synthetic financial data for the SuFinex Financial Intelligence Platform.

## **Objectives**

- Generate synthetic financial entities
- Maintain database relationships
- Simulate realistic customer behavior
- Generate transaction activity
- Simulate fraud, risk, and churn signals
- Load generated data into PostgreSQL

We're installing:

- Faker -> realistic names, cities, companies, etc.
- Pandas -> data manipulation
- NumPy -> numerical/random generation
- psycopg2-binary -> PostgreSQL connection
- SQLAlchemy -> database interaction  ->

In [2]:
!pip install faker pandas numpy psycopg2-binary sqlalchemy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 93.2 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
from faker import Faker
import random
from datetime import datetime, timedelta
import psycopg2
from sqlalchemy import create_engine, text

fake = Faker("en_IN")

# Reproducibility
np.random.seed(42)
Faker.seed(42)

print("SuFinex Synthetic Data Generator initialized successfully.")

SuFinex Synthetic Data Generator initialized successfully.


## Project Configuration

This section defines the scale, time period, and basic parameters used to generate the SuFinex synthetic dataset.

In [4]:
# ============================================================
# SuFinex Synthetic Data Configuration
# ============================================================

NUM_INSTITUTIONS = 5
NUM_CUSTOMERS = 10000
NUM_ACCOUNTS = 15000
NUM_CARDS = 12000
NUM_CREDIT_PROFILES = 10000
NUM_LOANS = 4000
NUM_MERCHANTS = 2000
NUM_PAYMENT_METHODS = 6
NUM_DEVICES = 12000
NUM_LOCATIONS = 500
NUM_SUPPORT_TICKETS = 5000
NUM_TRANSACTIONS = 100000

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

CURRENCY = "INR"

print("Configuration loaded successfully.")

Configuration loaded successfully.


In [5]:
# ============================================================
# ID Generation Helpers
# ============================================================


def generate_ids(prefix, count, width=6):
    """
    Generate unique IDs such as CUS000001, CUS000002, etc.
    """

    ids = []

    for i in range(1, count + 1):
        id_value = f"{prefix}{i:0{width}d}"
        ids.append(id_value)

    return ids


customer_ids = generate_ids("CUS", 5)

print(customer_ids)

['CUS000001', 'CUS000002', 'CUS000003', 'CUS000004', 'CUS000005']


## **1. Institution Data Generation**

Institutions represent financial organizations operating within the SuFinex platform.

In [17]:
# ============================================================
# Generate Institution Data
# ============================================================

# Generate unique institution IDs
institution_ids = generate_ids(
    "INS",
    NUM_INSTITUTIONS
)

# Synthetic institution names
institution_names = [
    "Apex Bank",
    "Nova Financial",
    "Prime Credit Union",
    "Urban Finance",
    "Vertex Payments"
]

# Institution types
institution_types = [
    "Bank",
    "Bank",
    "Credit Union",
    "NBFC",
    "Fintech"
]

# Headquarters
headquarters_cities = [
    "Mumbai",
    "Pune",
    "Bengaluru",
    "Delhi",
    "Hyderabad"
]

headquarters_states = [
    "Maharashtra",
    "Maharashtra",
    "Karnataka",
    "Delhi",
    "Telangana"
]

# Subscription plans
subscription_plans = [
    "Basic",
    "Standard",
    "Premium",
    "Premium",
    "Standard"
]

# Institution status
statuses = [
    "Active",
    "Active",
    "Active",
    "Active",
    "Active"
]

# Registration numbers
registration_numbers = [
    f"REG{i:08d}"
    for i in range(1, NUM_INSTITUTIONS + 1)
]

# Contact emails
contact_emails = [
    f"contact{i}@sufinex.com"
    for i in range(1, NUM_INSTITUTIONS + 1)
]

# Contact phone numbers
contact_phones = [
    f"+91-90000000{i:02d}"
    for i in range(1, NUM_INSTITUTIONS + 1)
]

# Onboarding dates
onboarding_dates = pd.date_range(
    start="2022-01-01",
    periods=NUM_INSTITUTIONS,
    freq="180D"
)

# Create Institution DataFrame
institutions_df = pd.DataFrame({
    "institution_id": institution_ids,
    "institution_name": institution_names,
    "institution_type": institution_types,
    "registration_number": registration_numbers,
    "headquarters_city": headquarters_cities,
    "headquarters_state": headquarters_states,
    "headquarters_country": ["India"] * NUM_INSTITUTIONS,
    "contact_email": contact_emails,
    "contact_phone": contact_phones,
    "onboarding_date": onboarding_dates,
    "subscription_plan": subscription_plans,
    "status": statuses
})

# Convert date to PostgreSQL-compatible format
institutions_df["onboarding_date"] = pd.to_datetime(
    institutions_df["onboarding_date"]
).dt.strftime("%Y-%m-%d")



In [18]:
institutions_df.head()

,institution_id,institution_name,institution_type,registration_number,headquarters_city,headquarters_state,headquarters_country,contact_email,contact_phone,onboarding_date,subscription_plan,status
0,INS000001,Apex Bank,Bank,REG00000001,Mumbai,Maharashtra,India,contact1@sufinex.com,+91-9000000001,2022-01-01,Basic,Active
1,INS000002,Nova Financial,Bank,REG00000002,Pune,Maharashtra,India,contact2@sufinex.com,+91-9000000002,2022-06-30,Standard,Active
2,INS000003,Prime Credit Union,Credit Union,REG00000003,Bengaluru,Karnataka,India,contact3@sufinex.com,+91-9000000003,2022-12-27,Premium,Active
3,INS000004,Urban Finance,NBFC,REG00000004,Delhi,Delhi,India,contact4@sufinex.com,+91-9000000004,2023-06-25,Premium,Active
4,INS000005,Vertex Payments,Fintech,REG00000005,Hyderabad,Telangana,India,contact5@sufinex.com,+91-9000000005,2023-12-22,Standard,Active


### **Validate the data**

In [19]:
# ============================================================
# Validate Institution Data
# ============================================================

expected_institution_columns = [
    "institution_id",
    "institution_name",
    "institution_type",
    "registration_number",
    "headquarters_city",
    "headquarters_state",
    "headquarters_country",
    "contact_email",
    "contact_phone",
    "onboarding_date",
    "subscription_plan",
    "status"
]

print("Columns:")
print(institutions_df.columns.tolist())

print("\nNumber of rows:")
print(len(institutions_df))

print("\nUnique institution IDs:")
print(institutions_df["institution_id"].nunique())

print("\nMissing values:")
print(institutions_df.isnull().sum())

Columns:
['institution_id', 'institution_name', 'institution_type', 'registration_number', 'headquarters_city', 'headquarters_state', 'headquarters_country', 'contact_email', 'contact_phone', 'onboarding_date', 'subscription_plan', 'status']

Number of rows:
5

Unique institution IDs:
5

Missing values:
institution_id          0
institution_name        0
institution_type        0
registration_number     0
headquarters_city       0
headquarters_state      0
headquarters_country    0
contact_email           0
contact_phone           0
onboarding_date         0
subscription_plan       0
status                  0
dtype: int64


## 2. **Customer Data Generation**

Customers represent individuals associated with financial institutions.

Customer attributes will be synthetically generated and will later support customer segmentation, risk analysis, churn prediction, and transaction behavior analysis.

### **Generate Customer IDs and Institution alignment**

In [21]:
# ============================================================
# Generate Customer IDs and Institution Assignments
# ============================================================

num_customers = NUM_CUSTOMERS

# Generate unique customer IDs
customer_ids = generate_ids(
    "CUS",
    num_customers
)

print("Number of customer IDs generated:", len(customer_ids))
print("First 5 IDs:", customer_ids[:5])

# Assign each customer to one of the synthetic institutions
customer_institution_ids = np.random.choice(
    institutions_df["institution_id"],
    size=num_customers
)

print("\nFirst 10 institution assignments:")
print(customer_institution_ids[:10])

Number of customer IDs generated: 10000
First 5 IDs: ['CUS000001', 'CUS000002', 'CUS000003', 'CUS000004', 'CUS000005']

First 10 institution assignments:
['INS000004' 'INS000005' 'INS000003' 'INS000005' 'INS000005' 'INS000002'
 'INS000003' 'INS000003' 'INS000003' 'INS000005']


### **Generate Indian cutomer names**

In [22]:
# ============================================================
# Generate Customer Names
# ============================================================

first_names = [
    fake.first_name()
    for _ in range(num_customers)
]

last_names = [
    fake.last_name()
    for _ in range(num_customers)
]

print("Sample customers:")

for i in range(5):
    print(first_names[i], last_names[i])

Sample customers:
Isaac Kota
Aryan Sodhi
Anvi Choudhary
Yash Bath
Udant Badal


### **Generate Customer Demographics**

In [23]:
# ============================================================
# Generate Customer Demographics
# ============================================================

date_of_birth = [
    fake.date_of_birth(
        minimum_age=18,
        maximum_age=75
    )
    for _ in range(num_customers)
]

genders = np.random.choice(
    ["Male", "Female", "Other"],
    size=num_customers,
    p=[0.48, 0.48, 0.04]
)

print("Sample dates of birth:", date_of_birth[:5])
print("Sample genders:", genders[:5])

Sample dates of birth: [datetime.date(2004, 3, 25), datetime.date(1984, 7, 18), datetime.date(1963, 6, 4), datetime.date(1996, 5, 20), datetime.date(1955, 10, 16)]
Sample genders: ['Male' 'Female' 'Male' 'Male' 'Male']


### **Generate Customer occupation and income**

In [24]:
# ============================================================
# Generate Occupation and Income
# ============================================================

occupations = [
    "Software Engineer",
    "Business Analyst",
    "Data Analyst",
    "Teacher",
    "Doctor",
    "Engineer",
    "Accountant",
    "Sales Executive",
    "Marketing Manager",
    "Government Employee",
    "Entrepreneur",
    "Consultant",
    "Student",
    "Self Employed"
]

customer_occupations = np.random.choice(
    occupations,
    size=num_customers
)

annual_incomes = np.random.lognormal(
    mean=np.log(600000),
    sigma=0.55,
    size=num_customers
)

annual_incomes = np.clip(
    annual_incomes,
    150000,
    5000000
)

annual_incomes = np.round(
    annual_incomes,
    2
)

print("Sample occupations:")
print(customer_occupations[:5])

print("\nSample annual incomes:")
print(annual_incomes[:5])

Sample occupations:
['Accountant' 'Government Employee' 'Student' 'Accountant' 'Teacher']

Sample annual incomes:
[1458761.15 1832907.19  473143.08  766575.48  940207.08]


### **Generate Customer since and status**

In [25]:
# ============================================================
# Generate Customer Lifecycle Data
# ============================================================

customer_since = [
    fake.date_between(
        start_date="-8y",
        end_date="today"
    )
    for _ in range(num_customers)
]

customer_status = np.random.choice(
    ["Active", "Inactive"],
    size=num_customers,
    p=[0.90, 0.10]
)

print("Sample customer_since dates:")
print(customer_since[:5])

print("\nCustomer status distribution:")
print(pd.Series(customer_status).value_counts())

Sample customer_since dates:
[datetime.date(2025, 6, 4), datetime.date(2024, 7, 22), datetime.date(2024, 10, 3), datetime.date(2021, 2, 15), datetime.date(2025, 9, 18)]

Customer status distribution:
Active      8940
Inactive    1060
Name: count, dtype: int64


### **Create the Customer DataFrame**

In [26]:
# ============================================================
# Create Customer DataFrame
# ============================================================

customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "institution_id": customer_institution_ids,
    "first_name": first_names,
    "last_name": last_names,
    "gender": genders,
    "date_of_birth": date_of_birth,
    "email": [
        f"customer{i}@sufinex.com"
        for i in range(1, num_customers + 1)
    ],
    "phone_number": [
        f"+91-9{np.random.randint(100000000, 999999999)}"
        for _ in range(num_customers)
    ],
    "occupation": customer_occupations,
    "annual_income": annual_incomes,
    "customer_since": customer_since,
    "customer_status": customer_status
})

customers_df.head()

,customer_id,institution_id,first_name,last_name,gender,date_of_birth,email,phone_number,occupation,annual_income,customer_since,customer_status
0,CUS000001,INS000004,Isaac,Kota,Male,2004-03-25,customer1@sufinex.com,+91-9648666688,Accountant,1458761.15,2025-06-04,Inactive
1,CUS000002,INS000005,Aryan,Sodhi,Female,1984-07-18,customer2@sufinex.com,+91-9199424356,Government Employee,1832907.19,2024-07-22,Active
2,CUS000003,INS000003,Anvi,Choudhary,Male,1963-06-04,customer3@sufinex.com,+91-9623576615,Student,473143.08,2024-10-03,Active
3,CUS000004,INS000005,Yash,Bath,Male,1996-05-20,customer4@sufinex.com,+91-9780827888,Accountant,766575.48,2021-02-15,Active
4,CUS000005,INS000005,Udant,Badal,Male,1955-10-16,customer5@sufinex.com,+91-9900147648,Teacher,940207.08,2025-09-18,Active


### **Validate the Customers data**

In [27]:
# ============================================================
# Validate Customer Data
# ============================================================

expected_customer_columns = [
    "customer_id",
    "institution_id",
    "first_name",
    "last_name",
    "gender",
    "date_of_birth",
    "email",
    "phone_number",
    "occupation",
    "annual_income",
    "customer_since",
    "customer_status"
]

print("Columns match PostgreSQL structure:",
      list(customers_df.columns) == expected_customer_columns)

print("\nNumber of customers:",
      len(customers_df))

print("\nNumber of columns:",
      len(customers_df.columns))

print("\nMissing values:")
print(customers_df.isnull().sum())

print("\nDuplicate customer IDs:",
      customers_df["customer_id"].duplicated().sum())

print("\nCustomers by institution:")
print(customers_df["institution_id"].value_counts())

Columns match PostgreSQL structure: True

Number of customers: 10000

Number of columns: 12

Missing values:
customer_id        0
institution_id     0
first_name         0
last_name          0
gender             0
date_of_birth      0
email              0
phone_number       0
occupation         0
annual_income      0
customer_since     0
customer_status    0
dtype: int64

Duplicate customer IDs: 0

Customers by institution:
institution_id
INS000001    2047
INS000005    2019
INS000002    2016
INS000004    1975
INS000003    1943
Name: count, dtype: int64


## 3. **Accounts Data Generation**

Accounts represent customer banking relationships, balances, and account lifecycle details.

In [29]:
# ============================================================
# Account Data Generation
# ============================================================

def generate_accounts(customers_df, institutions_df):
    """
    Generate synthetic customer bank accounts.
    """

    # Generate unique account IDs
    account_ids = generate_ids(
        "ACC",
        NUM_ACCOUNTS
    )

    # Assign each account to an existing customer
    account_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_ACCOUNTS
    )

    # Create a customer → institution mapping
    customer_institution_map = dict(
        zip(
            customers_df["customer_id"],
            customers_df["institution_id"]
        )
    )

    # Get the institution for each account's customer
    account_institution_ids = [
        customer_institution_map[customer_id]
        for customer_id in account_customer_ids
    ]

    # Generate account types
    account_types = np.random.choice(
        ["Savings", "Current", "Salary"],
        size=NUM_ACCOUNTS,
        p=[0.65, 0.20, 0.15]
    )

    # Generate unique-looking account numbers
    account_numbers = [
        fake.numerify(
            text="################"
        )
        for _ in range(NUM_ACCOUNTS)
    ]

    # Generate non-negative account balances
    balances = np.round(
        np.random.lognormal(
            mean=10,
            sigma=1,
            size=NUM_ACCOUNTS
        ),
        2
    )

    # Generate account opening dates
    opened_dates = [
        fake.date_between(
            start_date="-8y",
            end_date="today"
        )
        for _ in range(NUM_ACCOUNTS)
    ]

    # Generate account statuses
    account_statuses = np.random.choice(
        ["Active", "Inactive", "Closed"],
        size=NUM_ACCOUNTS,
        p=[0.85, 0.10, 0.05]
    )

    # Create Account DataFrame
    accounts_df = pd.DataFrame({
        "account_id": account_ids,
        "customer_id": account_customer_ids,
        "institution_id": account_institution_ids,
        "account_number": account_numbers,
        "account_type": account_types,
        "account_status": account_statuses,
        "account_open_date": opened_dates,
        "available_balance": balances
    })

    return accounts_df

In [30]:
# ============================================================
# Generate Account Data
# ============================================================

accounts_df = generate_accounts(
    customers_df,
    institutions_df
)

print("Accounts generated:", len(accounts_df))

accounts_df.head()

Accounts generated: 15000


,account_id,customer_id,institution_id,account_number,account_type,account_status,account_open_date,available_balance
0,ACC000001,CUS007144,INS000003,6849704241698142,Current,Active,2020-12-17,30579.81
1,ACC000002,CUS008080,INS000002,0768170068143659,Current,Active,2019-02-16,45543.65
2,ACC000003,CUS007215,INS000003,7239766105417150,Savings,Active,2024-05-17,59180.80
3,ACC000004,CUS000897,INS000001,0760175180077842,Current,Inactive,2020-01-03,38610.31
4,ACC000005,CUS007583,INS000004,8455761867706672,Salary,Active,2025-12-07,18744.81


## 4. **Cards Data Generation**

Cards store debit and credit card information linked to customer accounts.


In [33]:
# ============================================================
# Card Data Generation
# ============================================================

def generate_cards(accounts_df, customers_df):
    """
    Generate synthetic payment cards linked to customer accounts.
    """

    # Generate unique card IDs
    card_ids = generate_ids(
        "CARD",
        NUM_CARDS
    )

    # Assign each card to an existing account
    card_account_ids = np.random.choice(
        accounts_df["account_id"],
        size=NUM_CARDS
    )

    # Create account → customer mapping
    account_customer_map = dict(
        zip(
            accounts_df["account_id"],
            accounts_df["customer_id"]
        )
    )

    # Find the customer associated with each account
    card_customer_ids = [
        account_customer_map[account_id]
        for account_id in card_account_ids
    ]

    # Generate card types
    card_types = np.random.choice(
        ["Debit", "Credit"],
        size=NUM_CARDS,
        p=[0.70, 0.30]
    )

    # Generate card networks
    card_networks = np.random.choice(
        ["Visa", "Mastercard", "RuPay"],
        size=NUM_CARDS,
        p=[0.40, 0.35, 0.25]
    )

    # Generate card statuses
    card_statuses = np.random.choice(
        ["Active", "Blocked", "Expired"],
        size=NUM_CARDS,
        p=[0.90, 0.06, 0.04]
    )

    # Generate issue dates
    issue_dates = [
        fake.date_between(
            start_date="-5y",
            end_date="today"
        )
        for _ in range(NUM_CARDS)
    ]

    # Generate expiry dates four years after issue
    expiry_dates = [
        issue_date.replace(
            year=issue_date.year + 4
        )
        for issue_date in issue_dates
    ]

    # Create Card DataFrame
    cards_df = pd.DataFrame({
        "card_id": card_ids,
        "account_id": card_account_ids,
        "customer_id": card_customer_ids,
        "card_type": card_types,
        "network": card_networks,
        "issue_date": issue_dates,
        "expiry_date": expiry_dates,
        "card_status": card_statuses
    })

    return cards_df

In [34]:
# ============================================================
# Generate Card Data
# ============================================================

cards_df = generate_cards(
    accounts_df,
    customers_df
)

print("Cards generated:", len(cards_df))

cards_df.head()

Cards generated: 12000


,card_id,account_id,customer_id,card_type,network,issue_date,expiry_date,card_status
0,CARD000001,ACC000992,CUS000360,Debit,RuPay,2024-01-08,2028-01-08,Active
1,CARD000002,ACC004109,CUS004449,Debit,Visa,2024-09-05,2028-09-05,Active
2,CARD000003,ACC001455,CUS005049,Debit,Visa,2022-04-04,2026-04-04,Active
3,CARD000004,ACC002354,CUS007466,Debit,Visa,2025-08-11,2029-08-11,Active
4,CARD000005,ACC003655,CUS007525,Debit,Mastercard,2021-12-31,2025-12-31,Active


## 5. **Credit Profiles Data Generation**

Credit profiles capture customer creditworthiness, utilization, and repayment behavior indicators.

In [38]:
# ============================================================
# Credit Profile Data Generation
# ============================================================

def generate_credit_profiles(customers_df):
    """
    Generate one synthetic credit profile for each customer.
    """

    num_profiles = len(customers_df)

    # Generate unique credit profile IDs
    credit_profile_ids = generate_ids(
        "CRP",
        num_profiles
    )

    # One credit profile per customer
    customer_ids = customers_df["customer_id"].tolist()

    # Generate credit scores between 300 and 900
    credit_scores = np.random.randint(
        300,
        901,
        size=num_profiles
    )

    # Credit utilization percentage
    credit_utilization = np.round(
        np.random.uniform(
            0,
            95,
            size=num_profiles
        ),
        2
    )

    # Generate credit limits
    total_credit_limit = np.round(
        np.random.lognormal(
            mean=10.5,
            sigma=0.8,
            size=num_profiles
        ),
        2
    )

    # Calculate outstanding balance from utilization
    outstanding_balance = np.round(
        total_credit_limit *
        (credit_utilization / 100),
        2
    )

    # Number of previous delinquencies
    delinquency_count = np.random.poisson(
        lam=0.4,
        size=num_profiles
    )

    # Last profile update date
    last_updated = [
        fake.date_between(
            start_date="-1y",
            end_date="today"
        )
        for _ in range(num_profiles)
    ]

    # Create Credit Profile DataFrame
    credit_profiles_df = pd.DataFrame({
        "credit_profile_id": credit_profile_ids,
        "customer_id": customer_ids,
        "credit_score": credit_scores,
        "credit_utilization": credit_utilization,
        "total_credit_limit": total_credit_limit,
        "outstanding_balance": outstanding_balance,
        "delinquency_count": delinquency_count,
        "last_updated": last_updated
    })

    return credit_profiles_df

In [39]:
# ============================================================
# Generate Credit Profile Data
# ============================================================

credit_profiles_df = generate_credit_profiles(
    customers_df
)

print(
    "Credit profiles generated:",
    len(credit_profiles_df)
)

credit_profiles_df.head()

Credit profiles generated: 10000


,credit_profile_id,customer_id,credit_score,credit_utilization,total_credit_limit,outstanding_balance,delinquency_count,last_updated
0,CRP000001,CUS000001,676,91.10,19174.42,17467.90,0,2026-04-27
1,CRP000002,CUS000002,788,15.37,99532.08,15298.08,0,2026-04-15
2,CRP000003,CUS000003,822,94.06,42901.57,40353.22,0,2025-10-02
3,CRP000004,CUS000004,577,42.33,69066.34,29235.78,0,2025-11-29
4,CRP000005,CUS000005,312,31.89,44994.50,14348.75,1,2025-11-17


### **Validate Data**

In [40]:


print("\nAccounts:", len(accounts_df))
print("Cards:", len(cards_df))
print("Credit Profiles:", len(credit_profiles_df))

print("\nDuplicate Account IDs:",
      accounts_df["account_id"].duplicated().sum())

print("Duplicate Card IDs:",
      cards_df["card_id"].duplicated().sum())

print("Duplicate Credit Profile IDs:",
      credit_profiles_df["credit_profile_id"].duplicated().sum())

print("\nCredit Score Range:")
print(
    credit_profiles_df["credit_score"].min(),
    "to",
    credit_profiles_df["credit_score"].max()
)

print("\nMissing Values:")
print("\nAccounts:")
print(accounts_df.isnull().sum())

print("\nCards:")
print(cards_df.isnull().sum())

print("\nCredit Profiles:")
print(credit_profiles_df.isnull().sum())


Accounts: 15000
Cards: 12000
Credit Profiles: 10000

Duplicate Account IDs: 0
Duplicate Card IDs: 0
Duplicate Credit Profile IDs: 0

Credit Score Range:
300 to 900

Missing Values:

Accounts:
account_id           0
customer_id          0
institution_id       0
account_number       0
account_type         0
account_status       0
account_open_date    0
available_balance    0
dtype: int64

Cards:
card_id        0
account_id     0
customer_id    0
card_type      0
network        0
issue_date     0
expiry_date    0
card_status    0
dtype: int64

Credit Profiles:
credit_profile_id      0
customer_id            0
credit_score           0
credit_utilization     0
total_credit_limit     0
outstanding_balance    0
delinquency_count      0
last_updated           0
dtype: int64


## 6. **Loans Data Generation**

Loans track customer borrowing activities, loan amounts, tenure, and repayment status.

In [42]:
# ============================================================
# Loan Data Generation
# ============================================================

def generate_loans(customers_df):
    """
    Generate synthetic loans linked to existing customers.
    """

    # Generate unique loan IDs
    loan_ids = generate_ids(
        "LOAN",
        NUM_LOANS
    )

    # Select customers who will receive loans
    loan_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_LOANS,
        replace=False
    )

    # Create customer → institution mapping
    customer_institution_map = dict(
        zip(
            customers_df["customer_id"],
            customers_df["institution_id"]
        )
    )

    # Get institution for each loan customer
    loan_institution_ids = [
        customer_institution_map[customer_id]
        for customer_id in loan_customer_ids
    ]

    # Generate loan types
    loan_types = np.random.choice(
        ["Personal", "Home", "Auto", "Education"],
        size=NUM_LOANS,
        p=[0.45, 0.20, 0.20, 0.15]
    )

    # Generate original loan amounts
    loan_amounts = np.round(
        np.random.lognormal(
            mean=12,
            sigma=0.8,
            size=NUM_LOANS
        ),
        2
    )

    # Keep loan amounts within a realistic synthetic range
    loan_amounts = np.clip(
        loan_amounts,
        50000,
        5000000
    )

    loan_amounts = np.round(
        loan_amounts,
        2
    )

    # Generate outstanding amount
    outstanding_percentages = np.random.uniform(
        0,
        0.95,
        size=NUM_LOANS
    )

    outstanding_amounts = np.round(
        loan_amounts * outstanding_percentages,
        2
    )

    # Generate interest rates
    interest_rates = np.round(
        np.random.uniform(
            7.5,
            18.0,
            size=NUM_LOANS
        ),
        3
    )

    # Generate loan status
    loan_statuses = np.random.choice(
        ["Active", "Closed", "Defaulted"],
        size=NUM_LOANS,
        p=[0.75, 0.20, 0.05]
    )

    # Generate loan start dates
    loan_start_dates = [
        fake.date_between(
            start_date="-5y",
            end_date="today"
        )
        for _ in range(NUM_LOANS)
    ]

    # Generate loan tenure
    loan_tenures = np.random.choice(
        [12, 24, 36, 48, 60],
        size=NUM_LOANS
    )

    # Calculate end date from start date + tenure
    loan_end_dates = [
        start_date + pd.DateOffset(
            months=int(tenure)
        )
        for start_date, tenure
        in zip(loan_start_dates, loan_tenures)
    ]

    # Create Loan DataFrame
    loans_df = pd.DataFrame({
        "loan_id": loan_ids,
        "customer_id": loan_customer_ids,
        "institution_id": loan_institution_ids,
        "loan_type": loan_types,
        "loan_amount": loan_amounts,
        "outstanding_amount": outstanding_amounts,
        "interest_rate": interest_rates,
        "loan_status": loan_statuses,
        "start_date": loan_start_dates,
        "end_date": loan_end_dates
    })

    return loans_df

In [43]:
# ============================================================
# Generate Loan Data
# ============================================================

loans_df = generate_loans(
    customers_df
)

print(
    "Loans generated:",
    len(loans_df)
)

loans_df.head()

Loans generated: 4000


,loan_id,customer_id,institution_id,loan_type,loan_amount,outstanding_amount,interest_rate,loan_status,start_date,end_date
0,LOAN000001,CUS008900,INS000003,Auto,470404.39,235813.70,9.510,Active,2023-06-28,2025-06-28
1,LOAN000002,CUS009596,INS000004,Personal,258921.31,91494.74,13.253,Active,2025-08-06,2029-08-06
2,LOAN000003,CUS004324,INS000003,Personal,680357.36,269369.04,10.960,Active,2022-10-15,2024-10-15
3,LOAN000004,CUS003386,INS000003,Personal,143914.60,2411.10,12.796,Active,2026-03-29,2030-03-29
4,LOAN000005,CUS005047,INS000004,Auto,272688.31,40374.93,11.587,Active,2022-12-30,2023-12-30


## 7. **Merchants Data Generation**

Merchants represent businesses where customers perform financial transactions.

In [49]:
# ============================================================
# Indian City → State Mapping
# ============================================================

city_state_map = {
    "Mumbai": "Maharashtra",

    "Pune": "Maharashtra",

    "Thane": "Maharashtra",

    "Nagpur": "Maharashtra",

    "Bengaluru": "Karnataka",
    "Mysuru": "Karnataka",

    "Hyderabad": "Telangana",

    "Chennai": "Tamil Nadu",

    "Ahmedabad": "Gujarat",
    "Surat": "Gujarat",

    "Delhi": "Delhi",

    "Jaipur": "Rajasthan",

    "Kolkata": "West Bengal",

    "Kochi": "Kerala",

    "Lucknow": "Uttar Pradesh",

    "Indore": "Madhya Pradesh",
    "Bhopal": "Madhya Pradesh",

    "Patna": "Bihar",

    "Bhubaneswar": "Odisha",

    "Chandigarh": "Chandigarh"
}

# ============================================================
# City and State Lists
# ============================================================

cities_list = list(city_state_map.keys())

states_list = list(
    dict.fromkeys(city_state_map.values())
)

print("Number of cities:", len(cities_list))
print("Number of states:", len(states_list))

print("\nCities:")
print(cities_list)

print("\nStates:")
print(states_list)

Number of cities: 20
Number of states: 14

Cities:
['Mumbai', 'Pune', 'Thane', 'Nagpur', 'Bengaluru', 'Mysuru', 'Hyderabad', 'Chennai', 'Ahmedabad', 'Surat', 'Delhi', 'Jaipur', 'Kolkata', 'Kochi', 'Lucknow', 'Indore', 'Bhopal', 'Patna', 'Bhubaneswar', 'Chandigarh']

States:
['Maharashtra', 'Karnataka', 'Telangana', 'Tamil Nadu', 'Gujarat', 'Delhi', 'Rajasthan', 'West Bengal', 'Kerala', 'Uttar Pradesh', 'Madhya Pradesh', 'Bihar', 'Odisha', 'Chandigarh']


In [50]:
# ============================================================
# Merchant Data Generation
# ============================================================

def generate_merchants(institutions_df):
    """
    Generate synthetic merchants associated with institutions.
    """

    # Generate unique merchant IDs
    merchant_ids = generate_ids(
        "MER",
        NUM_MERCHANTS
    )

    # Assign each merchant to an existing institution
    merchant_institution_ids = np.random.choice(
        institutions_df["institution_id"],
        size=NUM_MERCHANTS
    )

    # Merchant categories
    merchant_categories = np.random.choice(
        [
            "Grocery",
            "Restaurant",
            "Fuel",
            "Travel",
            "Electronics",
            "Healthcare",
            "Shopping",
            "Entertainment"
        ],
        size=NUM_MERCHANTS
    )

    # Synthetic merchant names
    merchant_names = [
        f"{fake.company()} Store"
        for _ in range(NUM_MERCHANTS)
    ]

    # Indian cities
    cities = np.random.choice(
        cities_list,
        size=NUM_MERCHANTS
    )

    # Map city → state
    states = [
        city_state_map[city]
        for city in cities
    ]

    # Country
    countries = [
        "India"
    ] * NUM_MERCHANTS

    # Merchant status
    merchant_statuses = np.random.choice(
        ["Active", "Inactive"],
        size=NUM_MERCHANTS,
        p=[0.95, 0.05]
    )

    # Create DataFrame
    merchants_df = pd.DataFrame({
        "merchant_id": merchant_ids,
        "institution_id": merchant_institution_ids,
        "merchant_name": merchant_names,
        "merchant_category": merchant_categories,
        "city": cities,
        "state": states,
        "country": countries,
        "merchant_status": merchant_statuses
    })

    return merchants_df

In [51]:
# ============================================================
# Generate Merchant Data
# ============================================================

merchants_df = generate_merchants(
    institutions_df
)

print(
    "Merchants generated:",
    len(merchants_df)
)

merchants_df.head()

Merchants generated: 2000


,merchant_id,institution_id,merchant_name,merchant_category,city,state,country,merchant_status
0,MER000001,INS000001,Upadhyay PLC Store,Travel,Lucknow,Uttar Pradesh,India,Active
1,MER000002,INS000005,Goyal-Tailor Store,Travel,Kochi,Kerala,India,Active
2,MER000003,INS000003,Goswami-Shukla Store,Restaurant,Kolkata,West Bengal,India,Active
3,MER000004,INS000002,Rau-Luthra Store,Entertainment,Jaipur,Rajasthan,India,Active
4,MER000005,INS000005,Gour Inc Store,Shopping,Indore,Madhya Pradesh,India,Inactive


## 8. **Payment Methods Data Generation**

Payment methods define the available transaction channels used across the platform.

In [52]:
# ============================================================
# Payment Method Data Generation
# ============================================================

def generate_payment_methods():
    """
    Generate synthetic payment method reference data.
    """

    # Generate unique payment method IDs
    payment_method_ids = generate_ids(
        "PM",
        NUM_PAYMENT_METHODS
    )

    # Payment method names
    payment_methods = [
        "UPI",
        "Debit Card",
        "Credit Card",
        "Net Banking",
        "Wallet",
        "Bank Transfer"
    ]

    # Payment channels
    payment_channels = [
        "Digital",
        "Card",
        "Card",
        "Digital",
        "Digital",
        "Banking"
    ]

    # All methods are active
    payment_method_status = [
        "Active"
    ] * NUM_PAYMENT_METHODS

    # Create DataFrame
    payment_methods_df = pd.DataFrame({
        "payment_method_id": payment_method_ids,
        "payment_method": payment_methods,
        "payment_channel": payment_channels,
        "status": payment_method_status
    })

    return payment_methods_df

In [53]:
# ============================================================
# Generate Payment Method Data
# ============================================================

payment_methods_df = generate_payment_methods()

print(
    "Payment methods generated:",
    len(payment_methods_df)
)

payment_methods_df

Payment methods generated: 6


,payment_method_id,payment_method,payment_channel,status
0,PM000001,UPI,Digital,Active
1,PM000002,Debit Card,Card,Active
2,PM000003,Credit Card,Card,Active
3,PM000004,Net Banking,Digital,Active
4,PM000005,Wallet,Digital,Active
5,PM000006,Bank Transfer,Banking,Active



## 9. **Devices Data Generation**

Devices capture customer device information used to access financial services.

In [55]:
# ============================================================
# Device Data Generation
# ============================================================

def generate_devices():
    """
    Generate synthetic device reference data.
    """

    # Generate unique device IDs
    device_ids = generate_ids(
        "DEV",
        NUM_DEVICES
    )

    # Device types
    device_types = np.random.choice(
        ["Mobile", "Laptop", "Tablet"],
        size=NUM_DEVICES,
        p=[0.70, 0.20, 0.10]
    )

    # Operating systems
    operating_systems = np.random.choice(
        ["Android", "iOS", "Windows", "macOS"],
        size=NUM_DEVICES,
        p=[0.50, 0.25, 0.15, 0.10]
    )

    # Browsers
    browsers = np.random.choice(
        [
            "Chrome",
            "Safari",
            "Edge",
            "Firefox"
        ],
        size=NUM_DEVICES,
        p=[0.55, 0.20, 0.15, 0.10]
    )

    # Device status
    device_statuses = np.random.choice(
        ["Active", "Inactive"],
        size=NUM_DEVICES,
        p=[0.90, 0.10]
    )

    # Create DataFrame matching PostgreSQL
    devices_df = pd.DataFrame({
        "device_id": device_ids,
        "device_type": device_types,
        "operating_system": operating_systems,
        "browser": browsers,
        "device_status": device_statuses
    })

    return devices_df

In [56]:
# ============================================================
# Generate Device Data
# ============================================================

devices_df = generate_devices()

print(
    "Devices generated:",
    len(devices_df)
)

devices_df.head()

Devices generated: 12000


,device_id,device_type,operating_system,browser,device_status
0,DEV000001,Mobile,Android,Chrome,Active
1,DEV000002,Mobile,Android,Chrome,Active
2,DEV000003,Laptop,Android,Chrome,Active
3,DEV000004,Mobile,Android,Firefox,Active
4,DEV000005,Mobile,iOS,Chrome,Active



## 10: **Locations Data Generation**

Locations provide geographical reference data used for customer and transaction analytics.


In [57]:
# ============================================================
# Location Data Generation
# ============================================================

def generate_locations():
    """
    Generate synthetic Indian transaction locations.
    """

    # Generate unique location IDs
    location_ids = generate_ids(
        "LOC",
        NUM_LOCATIONS
    )

    # Randomly select Indian cities
    location_cities = np.random.choice(
        cities_list,
        size=NUM_LOCATIONS
    )

    # Map each city to its corresponding state
    location_states = [
        city_state_map[city]
        for city in location_cities
    ]

    # Country
    location_countries = [
        "India"
    ] * NUM_LOCATIONS

    # Synthetic geographic coordinates
    # Small variation around approximate city-center coordinates
    city_coordinates = {
        "Mumbai": (19.0760, 72.8777),
        "Pune": (18.5204, 73.8567),
        "Thane": (19.2183, 72.9781),
        "Nagpur": (21.1458, 79.0882),
        "Bengaluru": (12.9716, 77.5946),
        "Mysuru": (12.2958, 76.6394),
        "Hyderabad": (17.3850, 78.4867),
        "Chennai": (13.0827, 80.2707),
        "Ahmedabad": (23.0225, 72.5714),
        "Surat": (21.1702, 72.8311),
        "Delhi": (28.6139, 77.2090),
        "Jaipur": (26.9124, 75.7873),
        "Kolkata": (22.5726, 88.3639),
        "Kochi": (9.9312, 76.2673),
        "Lucknow": (26.8467, 80.9462),
        "Indore": (22.7196, 75.8577),
        "Bhopal": (23.2599, 77.4126),
        "Patna": (25.5941, 85.1376),
        "Bhubaneswar": (20.2961, 85.8245),
        "Chandigarh": (30.7333, 76.7794)
    }

    # Generate latitude and longitude
    latitudes = [
        round(
            city_coordinates[city][0] +
            np.random.uniform(-0.05, 0.05),
            6
        )
        for city in location_cities
    ]

    longitudes = [
        round(
            city_coordinates[city][1] +
            np.random.uniform(-0.05, 0.05),
            6
        )
        for city in location_cities
    ]

    # Create DataFrame matching PostgreSQL structure
    locations_df = pd.DataFrame({
        "location_id": location_ids,
        "city": location_cities,
        "state": location_states,
        "country": location_countries,
        "latitude": latitudes,
        "longitude": longitudes
    })

    return locations_df

In [58]:
# ============================================================
# Generate Location Data
# ============================================================

locations_df = generate_locations()

print(
    "Locations generated:",
    len(locations_df)
)

locations_df.head()

Locations generated: 500


,location_id,city,state,country,latitude,longitude
0,LOC000001,Mysuru,Karnataka,India,12.270772,76.667720
1,LOC000002,Kolkata,West Bengal,India,22.619316,88.354389
2,LOC000003,Kolkata,West Bengal,India,22.612217,88.339103
3,LOC000004,Hyderabad,Telangana,India,17.419135,78.496828
4,LOC000005,Kochi,Kerala,India,9.884743,76.281497


### **Validate Data**

In [59]:

print("\nLoans:", len(loans_df))
print("Merchants:", len(merchants_df))
print("Payment Methods:", len(payment_methods_df))
print("Devices:", len(devices_df))
print("Locations:", len(locations_df))

print("\nDuplicate IDs:")

print(
    "Loan IDs:",
    loans_df["loan_id"].duplicated().sum()
)

print(
    "Merchant IDs:",
    merchants_df["merchant_id"].duplicated().sum()
)

print(
    "Payment Method IDs:",
    payment_methods_df["payment_method_id"].duplicated().sum()
)

print(
    "Device IDs:",
    devices_df["device_id"].duplicated().sum()
)

print(
    "Location IDs:",
    locations_df["location_id"].duplicated().sum()
)


Loans: 4000
Merchants: 2000
Payment Methods: 6
Devices: 12000
Locations: 500

Duplicate IDs:
Loan IDs: 0
Merchant IDs: 0
Payment Method IDs: 0
Device IDs: 0
Location IDs: 0


## 11:**Support Tickets Data Generation**

Support tickets record customer service interactions, complaints, and issue resolution workflows.

In [60]:
# ============================================================
# Support Ticket Data Generation
# ============================================================

def generate_support_tickets(customers_df):
    """
    Generate synthetic customer support tickets.
    """

    # Generate unique ticket IDs
    ticket_ids = generate_ids(
        "TKT",
        NUM_SUPPORT_TICKETS
    )

    # Each ticket belongs to an existing customer
    ticket_customer_ids = np.random.choice(
        customers_df["customer_id"],
        size=NUM_SUPPORT_TICKETS
    )

    # Map customer → institution
    customer_institution_map = dict(
        zip(
            customers_df["customer_id"],
            customers_df["institution_id"]
        )
    )

    ticket_institution_ids = [
        customer_institution_map[customer_id]
        for customer_id in ticket_customer_ids
    ]

    # Issue categories
    issue_categories = np.random.choice(
        [
            "Transaction Issue",
            "Card Issue",
            "Account Issue",
            "Loan Query",
            "Fraud Report",
            "Payment Issue",
            "Technical Issue"
        ],
        size=NUM_SUPPORT_TICKETS,
        p=[
            0.20,
            0.15,
            0.15,
            0.10,
            0.10,
            0.20,
            0.10
        ]
    )

    # Ticket priority
    ticket_priorities = np.random.choice(
        ["Low", "Medium", "High", "Critical"],
        size=NUM_SUPPORT_TICKETS,
        p=[0.35, 0.40, 0.20, 0.05]
    )

    # Ticket status
    ticket_statuses = np.random.choice(
        ["Open", "In Progress", "Resolved", "Closed"],
        size=NUM_SUPPORT_TICKETS,
        p=[0.10, 0.10, 0.50, 0.30]
    )

    # Ticket creation dates
    created_dates = [
        fake.date_between(
            start_date="-2y",
            end_date="today"
        )
        for _ in range(NUM_SUPPORT_TICKETS)
    ]

    # Generate resolution dates only for resolved/closed tickets
    resolved_dates = []

    for status, created_date in zip(
        ticket_statuses,
        created_dates
    ):

        if status in ["Resolved", "Closed"]:

            resolution_days = np.random.randint(
                1,
                8
            )

            resolved_date = (
                pd.Timestamp(created_date)
                + pd.Timedelta(days=int(resolution_days))
            )

            resolved_dates.append(
                resolved_date.date()
            )

        else:
            resolved_dates.append(None)

    # Create DataFrame matching PostgreSQL
    support_tickets_df = pd.DataFrame({
        "ticket_id": ticket_ids,
        "customer_id": ticket_customer_ids,
        "institution_id": ticket_institution_ids,
        "issue_category": issue_categories,
        "priority": ticket_priorities,
        "status": ticket_statuses,
        "created_date": created_dates,
        "resolved_date": resolved_dates
    })

    return support_tickets_df

In [61]:
# ============================================================
# Generate Support Ticket Data
# ============================================================

support_tickets_df = generate_support_tickets(
    customers_df
)

print(
    "Support tickets generated:",
    len(support_tickets_df)
)

support_tickets_df.head()

Support tickets generated: 5000


,ticket_id,customer_id,institution_id,issue_category,priority,status,created_date,resolved_date
0,TKT000001,CUS005155,INS000002,Fraud Report,Medium,Closed,2025-12-16,2025-12-20
1,TKT000002,CUS000994,INS000002,Payment Issue,Medium,In Progress,2025-10-19,None
2,TKT000003,CUS005785,INS000001,Transaction Issue,Low,Resolved,2025-07-30,2025-08-02
3,TKT000004,CUS006600,INS000004,Transaction Issue,High,Resolved,2025-05-19,2025-05-25
4,TKT000005,CUS006204,INS000003,Transaction Issue,Medium,Resolved,2024-11-13,2024-11-16


## 12:**Transactions Data Generation**

Transactions store financial activity performed by customers across accounts, merchants, and payment channels.


In [ ]:
print("Accounts:")
print(accounts_df.columns.tolist())

print("\nMerchants:")
print(merchants_df.columns.tolist())

print("\nPayment Methods:")
print(payment_methods_df.columns.tolist())

print("\nDevices:")
print(devices_df.columns.tolist())

print("\nLocations:")
print(locations_df.columns.tolist())

Accounts:
['account_id', 'customer_id', 'institution_id', 'account_type', 'account_number', 'balance', 'opened_date', 'account_status']

Merchants:
['merchant_id', 'institution_id', 'merchant_name', 'merchant_category', 'city', 'merchant_status']

Payment Methods:
['payment_method_id', 'payment_method_name', 'status']

Devices:
['device_id', 'customer_id', 'device_type', 'operating_system', 'device_status']

Locations:
['location_id', 'city', 'state', 'country']


In [64]:
# ============================================================
# Transaction Data Generation
# ============================================================

def generate_transactions(
    customers_df,
    accounts_df,
    merchants_df,
    payment_methods_df,
    devices_df,
    locations_df
):
    """
    Generate synthetic financial transactions
    using existing reference data.
    """

    # --------------------------------------------------------
    # 1. Generate transaction IDs
    # --------------------------------------------------------

    transaction_ids = generate_ids(
        "TXN",
        NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 2. Select accounts
    # --------------------------------------------------------

    transaction_account_ids = np.random.choice(
        accounts_df["account_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 3. Map account → customer
    # --------------------------------------------------------

    account_customer_map = dict(
        zip(
            accounts_df["account_id"],
            accounts_df["customer_id"]
        )
    )

    transaction_customer_ids = [
        account_customer_map[account_id]
        for account_id in transaction_account_ids
    ]

    # --------------------------------------------------------
    # 4. Merchant
    # --------------------------------------------------------

    transaction_merchant_ids = np.random.choice(
        merchants_df["merchant_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 5. Payment method
    # --------------------------------------------------------

    transaction_payment_method_ids = np.random.choice(
        payment_methods_df["payment_method_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 6. Device
    # --------------------------------------------------------

    transaction_device_ids = np.random.choice(
        devices_df["device_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 7. Location
    # --------------------------------------------------------

    transaction_location_ids = np.random.choice(
        locations_df["location_id"],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 8. Transaction amount
    # --------------------------------------------------------

    transaction_amounts = np.round(
        np.random.lognormal(
            mean=7.5,
            sigma=1.2,
            size=NUM_TRANSACTIONS
        ),
        2
    )

    # Make sure all amounts are positive
    transaction_amounts = np.maximum(
        transaction_amounts,
        1.00
    )

    # --------------------------------------------------------
    # 9. Transaction types
    # --------------------------------------------------------

    transaction_types = np.random.choice(
        [
            "Purchase",
            "Withdrawal",
            "Transfer",
            "Bill Payment",
            "Refund"
        ],
        size=NUM_TRANSACTIONS,
        p=[
            0.55,
            0.15,
            0.15,
            0.10,
            0.05
        ]
    )

    # --------------------------------------------------------
    # 10. Transaction status
    # --------------------------------------------------------

    transaction_statuses = np.random.choice(
        [
            "Completed",
            "Pending",
            "Failed",
            "Reversed"
        ],
        size=NUM_TRANSACTIONS,
        p=[
            0.88,
            0.05,
            0.05,
            0.02
        ]
    )

    # --------------------------------------------------------
    # 11. Transaction timestamps
    # --------------------------------------------------------

    start_date = pd.Timestamp("2024-01-01")
    end_date = pd.Timestamp("2026-06-30")

    total_seconds = int(
        (end_date - start_date).total_seconds()
    )

    random_seconds = np.random.randint(
        0,
        total_seconds,
        size=NUM_TRANSACTIONS
    )

    transaction_timestamps = (
        start_date
        + pd.to_timedelta(
            random_seconds,
            unit="s"
        )
    )

    # --------------------------------------------------------
    # 12. Currency
    # --------------------------------------------------------

    currencies = [
        CURRENCY
    ] * NUM_TRANSACTIONS

    # --------------------------------------------------------
    # 13. Transaction description
    # --------------------------------------------------------

    descriptions = np.random.choice(
        [
            "Online Purchase",
            "ATM Withdrawal",
            "Fund Transfer",
            "Utility Bill Payment",
            "Merchant Payment",
            "Mobile Recharge",
            "Refund Transaction"
        ],
        size=NUM_TRANSACTIONS
    )

    # --------------------------------------------------------
    # 14. Create DataFrame
    # --------------------------------------------------------

    transactions_df = pd.DataFrame({

        "transaction_id": transaction_ids,

        "customer_id": transaction_customer_ids,

        "account_id": transaction_account_ids,

        "merchant_id": transaction_merchant_ids,

        "payment_method_id": transaction_payment_method_ids,

        "device_id": transaction_device_ids,

        "location_id": transaction_location_ids,

        "amount": transaction_amounts,

        "transaction_timestamp": transaction_timestamps,

        "transaction_type": transaction_types,

        "transaction_status": transaction_statuses,

        "currency": currencies,

        "description": descriptions
    })

    return transactions_df

In [65]:
# ============================================================
# Generate Transactions
# ============================================================

transactions_df = generate_transactions(
    customers_df,
    accounts_df,
    merchants_df,
    payment_methods_df,
    devices_df,
    locations_df
)

print(
    "Transactions generated:",
    len(transactions_df)
)

transactions_df.head()

Transactions generated: 100000


,transaction_id,customer_id,account_id,merchant_id,payment_method_id,device_id,location_id,amount,transaction_timestamp,transaction_type,transaction_status,currency,description
0,TXN000001,CUS006080,ACC012512,MER001654,PM000004,DEV003678,LOC000287,3771.15,2024-06-27 07:59:19,Withdrawal,Completed,INR,Utility Bill Payment
1,TXN000002,CUS006008,ACC006415,MER000518,PM000006,DEV006708,LOC000469,2816.00,2025-02-23 11:01:30,Bill Payment,Completed,INR,Merchant Payment
2,TXN000003,CUS005654,ACC009652,MER001836,PM000005,DEV007572,LOC000048,34946.81,2024-07-27 12:21:14,Bill Payment,Pending,INR,Online Purchase
3,TXN000004,CUS006970,ACC005132,MER001621,PM000003,DEV005219,LOC000003,1808.30,2025-07-07 03:48:59,Withdrawal,Failed,INR,Utility Bill Payment
4,TXN000005,CUS008873,ACC013531,MER000195,PM000005,DEV003562,LOC000373,35768.40,2024-04-10 23:55:38,Purchase,Completed,INR,ATM Withdrawal


In [66]:
# ============================================================
# Validate Transaction Structure
# ============================================================

expected_transaction_columns = [
    "transaction_id",
    "customer_id",
    "account_id",
    "merchant_id",
    "payment_method_id",
    "device_id",
    "location_id",
    "amount",
    "transaction_timestamp",
    "transaction_type",
    "transaction_status",
    "currency",
    "description"
]

print(
    "Columns match PostgreSQL structure:",
    list(transactions_df.columns)
    == expected_transaction_columns
)

print(
    "\nNumber of rows:",
    len(transactions_df)
)

print(
    "Number of columns:",
    len(transactions_df.columns)
)

print(
    "\nDuplicate transaction IDs:",
    transactions_df["transaction_id"].duplicated().sum()
)

print(
    "\nMissing values:"
)

print(
    transactions_df.isnull().sum()
)

Columns match PostgreSQL structure: True

Number of rows: 100000
Number of columns: 13

Duplicate transaction IDs: 0

Missing values:
transaction_id           0
customer_id              0
account_id               0
merchant_id              0
payment_method_id        0
device_id                0
location_id              0
amount                   0
transaction_timestamp    0
transaction_type         0
transaction_status       0
currency                 0
description              0
dtype: int64


### **Validate data**

In [68]:
print("========== TRANSACTION VALIDATION ==========")

print("\nNumber of transactions:")
print(len(transactions_df))

print("\nNumber of columns:")
print(len(transactions_df.columns))

print("\nDuplicate Transaction IDs:")
print(
    transactions_df["transaction_id"].duplicated().sum()
)

print("\nMissing Values:")
print(
    transactions_df.isnull().sum()
)

========== TRANSACTION VALIDATION ==========

Number of transactions:
100000

Number of columns:
13

Duplicate Transaction IDs:
0

Missing Values:
transaction_id           0
customer_id              0
account_id               0
merchant_id              0
payment_method_id        0
device_id                0
location_id              0
amount                   0
transaction_timestamp    0
transaction_type         0
transaction_status       0
currency                 0
description              0
dtype: int64


## 13:**Fraud Predictions Data Generation**

Fraud predictions capture transaction-level fraud risk assessments generated using behavioral risk signals.

In [69]:
# ============================================================
# Fraud Prediction Data Generation
# ============================================================

def generate_fraud_predictions(transactions_df):
    """
    Generate synthetic fraud predictions using
    transaction-level behavioral risk signals.

    Target:
    98,000 Legitimate
    2,000 Fraud
    """

    # --------------------------------------------------------
    # 1. Generate Fraud Prediction IDs
    # --------------------------------------------------------

    fraud_prediction_ids = generate_ids(
        "FRD",
        len(transactions_df)
    )

    # --------------------------------------------------------
    # 2. Start with base risk score
    # --------------------------------------------------------

    fraud_risk_score = np.zeros(
        len(transactions_df)
    )

    # --------------------------------------------------------
    # 3. High transaction amount
    # --------------------------------------------------------

    high_amount_threshold = transactions_df[
        "amount"
    ].quantile(0.95)

    high_amount = (
        transactions_df["amount"]
        >= high_amount_threshold
    )

    fraud_risk_score += (
        high_amount.astype(int) * 25
    )

    # --------------------------------------------------------
    # 4. Failed transactions
    # --------------------------------------------------------

    failed_transaction = (
        transactions_df["transaction_status"]
        == "Failed"
    )

    fraud_risk_score += (
        failed_transaction.astype(int) * 15
    )

    # --------------------------------------------------------
    # 5. Reversed transactions
    # --------------------------------------------------------

    reversed_transaction = (
        transactions_df["transaction_status"]
        == "Reversed"
    )

    fraud_risk_score += (
        reversed_transaction.astype(int) * 15
    )

    # --------------------------------------------------------
    # 6. Withdrawal transactions
    # --------------------------------------------------------

    withdrawal = (
        transactions_df["transaction_type"]
        == "Withdrawal"
    )

    fraud_risk_score += (
        withdrawal.astype(int) * 10
    )

    # --------------------------------------------------------
    # 7. Customer transaction behavior
    # --------------------------------------------------------

    customer_average_amount = (
        transactions_df
        .groupby("customer_id")["amount"]
        .transform("mean")
    )

    unusually_large_transaction = (
        transactions_df["amount"]
        > customer_average_amount * 3
    )

    fraud_risk_score += (
        unusually_large_transaction.astype(int) * 20
    )

    # --------------------------------------------------------
    # 8. Random variation
    # --------------------------------------------------------

    fraud_risk_score += np.random.uniform(
        0,
        10,
        size=len(transactions_df)
    )

    # --------------------------------------------------------
    # 9. Keep risk score between 0 and 100
    # --------------------------------------------------------

    fraud_risk_score = np.clip(
        fraud_risk_score,
        0,
        100
    )

    fraud_risk_score = np.round(
        fraud_risk_score,
        2
    )

    # --------------------------------------------------------
    # 10. Risk level
    # --------------------------------------------------------

    risk_level = pd.cut(
        fraud_risk_score,
        bins=[-1, 30, 60, 100],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    )

    risk_level = risk_level.astype(str)

    # --------------------------------------------------------
    # 11. Select top 2% as fraud
    # --------------------------------------------------------

    fraud_cutoff = np.percentile(
        fraud_risk_score,
        98
    )

    fraud_prediction = np.where(
        fraud_risk_score >= fraud_cutoff,
        "Fraud",
        "Legitimate"
    )

    # --------------------------------------------------------
    # 12. Fraud probability
    # --------------------------------------------------------

    # Convert risk score into probability between 0 and 1

    fraud_probability = (
        fraud_risk_score / 100
    )

    fraud_probability = np.round(
        fraud_probability,
        5
    )

    # --------------------------------------------------------
    # 13. Prediction timestamp
    # --------------------------------------------------------

    prediction_timestamp = (
        transactions_df[
            "transaction_timestamp"
        ]
        + pd.to_timedelta(
            np.random.randint(
                1,
                300,
                size=len(transactions_df)
            ),
            unit="s"
        )
    )

    # --------------------------------------------------------
    # 14. Model information
    # --------------------------------------------------------

    model_names = [
        "SuFinex Fraud Detection Model"
    ] * len(transactions_df)

    model_versions = [
        "1.0"
    ] * len(transactions_df)

    # --------------------------------------------------------
    # 15. Create PostgreSQL-compatible DataFrame
    # --------------------------------------------------------

    fraud_predictions_df = pd.DataFrame({

        "fraud_prediction_id":
            fraud_prediction_ids,

        "transaction_id":
            transactions_df["transaction_id"],

        "fraud_prediction":
            fraud_prediction,

        "fraud_probability":
            fraud_probability,

        "risk_level":
            risk_level,

        "model_name":
            model_names,

        "model_version":
            model_versions,

        "prediction_timestamp":
            prediction_timestamp
    })

    return fraud_predictions_df

In [70]:
# ============================================================
# Generate Fraud Predictions
# ============================================================

fraud_predictions_df = generate_fraud_predictions(
    transactions_df
)

print(
    "Fraud predictions generated:",
    len(fraud_predictions_df)
)

fraud_predictions_df.head()

Fraud predictions generated: 100000


,fraud_prediction_id,transaction_id,fraud_prediction,fraud_probability,risk_level,model_name,model_version,prediction_timestamp
0,FRD000001,TXN000001,Legitimate,0.1573,Low,SuFinex Fraud Detection Model,1.0,2024-06-27 08:03:29
1,FRD000002,TXN000002,Legitimate,0.0452,Low,SuFinex Fraud Detection Model,1.0,2025-02-23 11:01:53
2,FRD000003,TXN000003,Legitimate,0.4998,Medium,SuFinex Fraud Detection Model,1.0,2024-07-27 12:21:24
3,FRD000004,TXN000004,Legitimate,0.3294,Medium,SuFinex Fraud Detection Model,1.0,2025-07-07 03:52:37
4,FRD000005,TXN000005,Legitimate,0.4907,Medium,SuFinex Fraud Detection Model,1.0,2024-04-10 23:58:39


In [71]:
fraud_predictions_df[
    "fraud_prediction"
].value_counts()

,count
fraud_prediction,
Legitimate,97998
Fraud,2002


In [72]:
fraud_predictions_df[
    fraud_predictions_df["fraud_prediction"] == "Fraud"
].head(20)

,fraud_prediction_id,transaction_id,fraud_prediction,fraud_probability,risk_level,model_name,model_version,prediction_timestamp
10,FRD000011,TXN000011,Fraud,0.5380,Medium,SuFinex Fraud Detection Model,1.0,2024-02-17 15:06:21
22,FRD000023,TXN000023,Fraud,0.5825,Medium,SuFinex Fraud Detection Model,1.0,2024-10-21 00:12:04
30,FRD000031,TXN000031,Fraud,0.5426,Medium,SuFinex Fraud Detection Model,1.0,2026-06-01 18:25:25
66,FRD000067,TXN000067,Fraud,0.6283,High,SuFinex Fraud Detection Model,1.0,2024-09-22 08:38:21
128,FRD000129,TXN000129,Fraud,0.5213,Medium,SuFinex Fraud Detection Model,1.0,2025-07-11 16:22:00
136,FRD000137,TXN000137,Fraud,0.6226,High,SuFinex Fraud Detection Model,1.0,2025-06-05 06:01:19
138,FRD000139,TXN000139,Fraud,0.5951,Medium,SuFinex Fraud Detection Model,1.0,2025-12-20 10:55:27
239,FRD000240,TXN000240,Fraud,0.5335,Medium,SuFinex Fraud Detection Model,1.0,2026-02-03 02:36:34
411,FRD000412,TXN000412,Fraud,0.6498,High,SuFinex Fraud Detection Model,1.0,2024-04-09 00:07:51
479,FRD000480,TXN000480,Fraud,0.5384,Medium,SuFinex Fraud Detection Model,1.0,2025-05-04 08:56:26



## 14:**Risk Scores Data Generation**

Risk scores provide customer-level financial risk assessments based on credit and behavioral indicators.

In [75]:
# ============================================================
# Risk Score Data Generation
# ============================================================

def generate_risk_scores(
    customers_df,
    credit_profiles_df,
    transactions_df,
    support_tickets_df
):
    """
    Generate customer-level synthetic risk scores
    using financial and behavioral indicators.
    """

    # --------------------------------------------------------
    # 1. Customer IDs
    # --------------------------------------------------------

    customer_ids = customers_df["customer_id"].tolist()

    # --------------------------------------------------------
    # 2. Credit information
    # --------------------------------------------------------

    credit_data = credit_profiles_df[
        [
            "customer_id",
            "credit_score",
            "credit_utilization"
        ]
    ].copy()

    # Lower credit score = higher risk
    credit_data["credit_risk"] = (
        100
        - (
            (credit_data["credit_score"] - 300)
            / 600
            * 100
        )
    )

    # Higher utilization = higher risk
    credit_data["utilization_risk"] = (
        credit_data["credit_utilization"]
        .clip(0, 100)
    )

    # --------------------------------------------------------
    # 3. Transaction behavior
    # --------------------------------------------------------

    transaction_summary = (
        transactions_df
        .groupby("customer_id")
        .agg(
            transaction_count=(
                "transaction_id",
                "count"
            ),
            failed_count=(
                "transaction_status",
                lambda x: (x == "Failed").sum()
            ),
            reversed_count=(
                "transaction_status",
                lambda x: (x == "Reversed").sum()
            )
        )
        .reset_index()
    )

    transaction_summary["failed_rate"] = (
        transaction_summary["failed_count"]
        / transaction_summary["transaction_count"]
        * 100
    )

    transaction_summary["reversed_rate"] = (
        transaction_summary["reversed_count"]
        / transaction_summary["transaction_count"]
        * 100
    )

    # --------------------------------------------------------
    # 4. Support ticket activity
    # --------------------------------------------------------

    ticket_summary = (
        support_tickets_df
        .groupby("customer_id")
        .size()
        .reset_index(
            name="support_ticket_count"
        )
    )

    # --------------------------------------------------------
    # 5. Combine risk indicators
    # --------------------------------------------------------

    risk_data = (
        pd.DataFrame({
            "customer_id": customer_ids
        })
        .merge(
            credit_data,
            on="customer_id",
            how="left"
        )
        .merge(
            transaction_summary,
            on="customer_id",
            how="left"
        )
        .merge(
            ticket_summary,
            on="customer_id",
            how="left"
        )
    )

    # Customers without activity receive zero
    risk_data = risk_data.fillna(0)

    # --------------------------------------------------------
    # 6. Calculate overall risk score
    # --------------------------------------------------------

    risk_data["risk_score"] = (
        risk_data["credit_risk"] * 0.35
        + risk_data["utilization_risk"] * 0.25
        + risk_data["failed_rate"] * 0.05
        + risk_data["reversed_rate"] * 0.05
        + risk_data["support_ticket_count"] * 2
    )

    # --------------------------------------------------------
    # 7. Keep score between 0 and 100
    # --------------------------------------------------------

    risk_data["risk_score"] = np.clip(
        risk_data["risk_score"],
        0,
        100
    )

    risk_data["risk_score"] = np.round(
        risk_data["risk_score"],
        2
    )

    # --------------------------------------------------------
    # 8. Risk category
    # --------------------------------------------------------

    risk_data["risk_category"] = pd.cut(
        risk_data["risk_score"],
        bins=[-1, 30, 60, 100],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    ).astype(str)

    # --------------------------------------------------------
    # 9. Score timestamp
    # --------------------------------------------------------

    score_timestamp = pd.Timestamp(
        "2026-06-30"
    )

    # --------------------------------------------------------
    # 10. Model information
    # --------------------------------------------------------

    model_name = [
        "SuFinex Customer Risk Model"
    ] * len(risk_data)

    model_version = [
        "1.0"
    ] * len(risk_data)

    # --------------------------------------------------------
    # 11. Create PostgreSQL-compatible DataFrame
    # --------------------------------------------------------

    risk_scores_df = pd.DataFrame({

        "risk_score_id": generate_ids(
            "RSK",
            len(risk_data)
        ),

        "customer_id":
            risk_data["customer_id"],

        "risk_score":
            risk_data["risk_score"],

        "risk_category":
            risk_data["risk_category"],

        "model_name":
            model_name,

        "model_version":
            model_version,

        "score_timestamp":
            score_timestamp
    })

    return risk_scores_df

In [76]:
# ============================================================
# Generate Risk Scores
# ============================================================

risk_scores_df = generate_risk_scores(
    customers_df,
    credit_profiles_df,
    transactions_df,
    support_tickets_df
)

print(
    "Risk scores generated:",
    len(risk_scores_df)
)

risk_scores_df.head()

Risk scores generated: 10000


,risk_score_id,customer_id,risk_score,risk_category,model_name,model_version,score_timestamp
0,RSK000001,CUS000001,37.84,Medium,SuFinex Customer Risk Model,1.0,2026-06-30
1,RSK000002,CUS000002,11.17,Low,SuFinex Customer Risk Model,1.0,2026-06-30
2,RSK000003,CUS000003,30.06,Medium,SuFinex Customer Risk Model,1.0,2026-06-30
3,RSK000004,CUS000004,33.88,Medium,SuFinex Customer Risk Model,1.0,2026-06-30
4,RSK000005,CUS000005,42.27,Medium,SuFinex Customer Risk Model,1.0,2026-06-30


In [77]:
risk_scores_df[
    "risk_category"
].value_counts()

,count
risk_category,
Medium,5341
Low,4649
High,10


## 15:**Churn Predictions Data Generation**

Churn predictions estimate the likelihood of customers discontinuing their relationship with the institution.

In [78]:
def generate_churn_predictions(
    customers_df,
    transactions_df,
    support_tickets_df,
    risk_scores_df
):
    """
    Generate synthetic customer-level churn predictions
    matching the PostgreSQL churn_prediction table.
    """

    # --------------------------------------------------------
    # 1. Customer IDs
    # --------------------------------------------------------

    customer_ids = customers_df["customer_id"].tolist()

    # --------------------------------------------------------
    # 2. Transaction behavior
    # --------------------------------------------------------

    transaction_summary = (
        transactions_df
        .groupby("customer_id")
        .agg(
            transaction_count=(
                "transaction_id",
                "count"
            ),
            last_transaction=(
                "transaction_timestamp",
                "max"
            )
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 3. Calculate recency
    # --------------------------------------------------------

    analysis_date = transactions_df[
        "transaction_timestamp"
    ].max()

    transaction_summary["days_since_transaction"] = (
        analysis_date
        - transaction_summary["last_transaction"]
    ).dt.days

    # --------------------------------------------------------
    # 4. Support ticket activity
    # --------------------------------------------------------

    ticket_summary = (
        support_tickets_df
        .groupby("customer_id")
        .size()
        .reset_index(
            name="support_ticket_count"
        )
    )

    # --------------------------------------------------------
    # 5. Customer risk score
    # --------------------------------------------------------

    risk_data = risk_scores_df[
        [
            "customer_id",
            "risk_score"
        ]
    ].copy()

    # --------------------------------------------------------
    # 6. Combine all customer-level information
    # --------------------------------------------------------

    churn_data = (
        pd.DataFrame({
            "customer_id": customer_ids
        })
        .merge(
            transaction_summary,
            on="customer_id",
            how="left"
        )
        .merge(
            ticket_summary,
            on="customer_id",
            how="left"
        )
        .merge(
            risk_data,
            on="customer_id",
            how="left"
        )
    )

    # --------------------------------------------------------
    # 7. Handle missing values
    # --------------------------------------------------------

    churn_data["transaction_count"] = (
        churn_data["transaction_count"].fillna(0)
    )

    churn_data["days_since_transaction"] = (
        churn_data["days_since_transaction"].fillna(365)
    )

    churn_data["support_ticket_count"] = (
        churn_data["support_ticket_count"].fillna(0)
    )

    churn_data["risk_score"] = (
        churn_data["risk_score"].fillna(0)
    )

    # --------------------------------------------------------
    # 8. Calculate churn score
    # --------------------------------------------------------

    churn_data["churn_score"] = (
        churn_data["days_since_transaction"] * 0.35
        + churn_data["support_ticket_count"] * 5
        + churn_data["risk_score"] * 0.20
    )

    # --------------------------------------------------------
    # 9. Normalize churn score to 0-100
    # --------------------------------------------------------

    churn_min = churn_data["churn_score"].min()
    churn_max = churn_data["churn_score"].max()

    if churn_max > churn_min:

        churn_data["churn_score"] = (
            (
                churn_data["churn_score"]
                - churn_min
            )
            / (churn_max - churn_min)
            * 100
        )

    else:

        churn_data["churn_score"] = 0

    churn_data["churn_score"] = np.round(
        churn_data["churn_score"],
        2
    )

    # --------------------------------------------------------
    # 10. Churn risk level
    # --------------------------------------------------------

    churn_data["churn_risk_level"] = pd.cut(
        churn_data["churn_score"],
        bins=[-1, 30, 60, 100],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    ).astype(str)

    # --------------------------------------------------------
    # 11. Convert churn score to probability
    # --------------------------------------------------------

    churn_data["churn_probability"] = (
        churn_data["churn_score"] / 100
    )

    churn_data["churn_probability"] = np.round(
        churn_data["churn_probability"],
        5
    )

    # --------------------------------------------------------
    # 12. Churn prediction
    # PostgreSQL CHECK constraint allows only:
    # "Churn" or "Not Churn"
    # --------------------------------------------------------

    churn_data["churn_prediction"] = np.where(
        churn_data["churn_probability"] >= 0.60,
        "Churn",
        "Not Churn"
    )

    # --------------------------------------------------------
    # 13. Model information
    # --------------------------------------------------------

    model_name = "Synthetic Churn Risk Model"
    model_version = "1.0"

    # --------------------------------------------------------
    # 14. Prediction timestamp
    # --------------------------------------------------------

    prediction_timestamp = pd.Timestamp(
        "2026-06-30"
    )

    # --------------------------------------------------------
    # 15. Create final DataFrame
    # Matching PostgreSQL table exactly
    # --------------------------------------------------------

    churn_predictions_df = pd.DataFrame({

        "churn_prediction_id": generate_ids(
            "CHN",
            len(churn_data)
        ),

        "customer_id": churn_data[
            "customer_id"
        ],

        "churn_prediction": churn_data[
            "churn_prediction"
        ],

        "churn_probability": churn_data[
            "churn_probability"
        ],

        "churn_risk_level": churn_data[
            "churn_risk_level"
        ],

        "model_name": model_name,

        "model_version": model_version,

        "prediction_timestamp": prediction_timestamp
    })

    return churn_predictions_df




Churn predictions generated: 10000


,churn_prediction_id,customer_id,churn_prediction,churn_probability,churn_risk_level,model_name,model_version,prediction_timestamp
0,CHN000001,CUS000001,Not Churn,0.0848,Low,Synthetic Churn Risk Model,1.0,2026-06-30
1,CHN000002,CUS000002,Not Churn,0.0057,Low,Synthetic Churn Risk Model,1.0,2026-06-30
2,CHN000003,CUS000003,Not Churn,0.2619,Low,Synthetic Churn Risk Model,1.0,2026-06-30
3,CHN000004,CUS000004,Not Churn,0.1322,Low,Synthetic Churn Risk Model,1.0,2026-06-30
4,CHN000005,CUS000005,Not Churn,0.1862,Low,Synthetic Churn Risk Model,1.0,2026-06-30


In [79]:
# ============================================================
# Generate Churn Predictions
# ============================================================

churn_predictions_df = generate_churn_predictions(
    customers_df,
    transactions_df,
    support_tickets_df,
    risk_scores_df
)

print(
    "Churn predictions generated:",
    len(churn_predictions_df)
)

churn_predictions_df.head()

Churn predictions generated: 10000


,churn_prediction_id,customer_id,churn_prediction,churn_probability,churn_risk_level,model_name,model_version,prediction_timestamp
0,CHN000001,CUS000001,Not Churn,0.0848,Low,Synthetic Churn Risk Model,1.0,2026-06-30
1,CHN000002,CUS000002,Not Churn,0.0057,Low,Synthetic Churn Risk Model,1.0,2026-06-30
2,CHN000003,CUS000003,Not Churn,0.2619,Low,Synthetic Churn Risk Model,1.0,2026-06-30
3,CHN000004,CUS000004,Not Churn,0.1322,Low,Synthetic Churn Risk Model,1.0,2026-06-30
4,CHN000005,CUS000005,Not Churn,0.1862,Low,Synthetic Churn Risk Model,1.0,2026-06-30


In [80]:
churn_predictions_df['churn_risk_level'].value_counts()

,count
churn_risk_level,
Low,7221
Medium,2713
High,66


## 16:**Fraud Investigations Data Generation**

Fraud investigations track the review and resolution process for transactions flagged as potentially fraudulent.


In [87]:
def generate_fraud_investigations(
    fraud_predictions_df
):
    """
    Generate synthetic fraud investigations
    for transactions predicted as fraudulent.
    """

    # --------------------------------------------------------
    # 1. Select fraudulent transactions
    # --------------------------------------------------------

    fraud_cases = fraud_predictions_df[
        fraud_predictions_df["fraud_prediction"] == "Fraud"
    ].copy()

    fraud_cases = fraud_cases.reset_index(drop=True)

    investigation_count = len(fraud_cases)

    print(
        "Fraud cases selected for investigation:",
        investigation_count
    )

    # --------------------------------------------------------
    # 2. Investigation status
    # PostgreSQL allowed values:
    # Open, In Progress, Resolved, Closed
    # --------------------------------------------------------

    investigation_status = np.random.choice(
        [
            "Open",
            "In Progress",
            "Resolved",
            "Closed"
        ],
        size=investigation_count,
        p=[0.15, 0.25, 0.35, 0.25]
    )

    # --------------------------------------------------------
    # 3. Investigation priority
    # Based on fraud risk level
    # --------------------------------------------------------

    investigation_priority = np.select(
        [
            fraud_cases["risk_level"] == "High",
            fraud_cases["risk_level"] == "Medium"
        ],
        [
            "Critical",
            "High"
        ],
        default="Medium"
    )

    # --------------------------------------------------------
    # 4. Investigator names
    # --------------------------------------------------------

    investigator_names = [
        fake.name()
        for _ in range(investigation_count)
    ]

    # --------------------------------------------------------
    # 5. Investigation notes
    # --------------------------------------------------------

    investigation_notes = np.select(
        [
            fraud_cases["risk_level"] == "High",
            fraud_cases["risk_level"] == "Medium"
        ],
        [
            "High-risk fraudulent transaction requires immediate investigation.",
            "Medium-risk fraudulent transaction flagged for detailed review."
        ],
        default="Fraudulent transaction flagged for investigation."
    )

    # --------------------------------------------------------
    # 6. Investigation result
    # Only completed investigations get a result
    # --------------------------------------------------------

    investigation_result = []

    for status in investigation_status:

        if status in ["Resolved", "Closed"]:

            result = np.random.choice(
                [
                    "Confirmed Fraud",
                    "False Positive"
                ],
                p=[0.70, 0.30]
            )

            investigation_result.append(result)

        else:

            investigation_result.append(None)

    # --------------------------------------------------------
    # 7. Created timestamp
    # --------------------------------------------------------

    created_at = (
        fraud_cases["prediction_timestamp"]
        + pd.to_timedelta(
            np.random.randint(
                1,
                86400,
                size=investigation_count
            ),
            unit="s"
        )
    )

    # --------------------------------------------------------
    # 8. Resolved timestamp
    # Only Resolved / Closed cases get a timestamp
    # --------------------------------------------------------

    resolved_at = []

    for i in range(investigation_count):

        if investigation_status[i] in [
            "Resolved",
            "Closed"
        ]:

            resolution_time = (
                created_at.iloc[i]
                + pd.Timedelta(
                    days=np.random.randint(1, 8)
                )
            )

            resolved_at.append(
                resolution_time
            )

        else:

            resolved_at.append(None)

    # --------------------------------------------------------
    # 9. Create final DataFrame
    # Exactly matches PostgreSQL table
    # --------------------------------------------------------

    fraud_investigations_df = pd.DataFrame({

        "investigation_id": generate_ids(
            "INV",
            investigation_count
        ),

        "transaction_id": fraud_cases[
            "transaction_id"
        ].values,

        "fraud_prediction_id": fraud_cases[
            "fraud_prediction_id"
        ].values,

        "investigation_status": investigation_status,

        "investigation_priority": investigation_priority,

        "investigator_name": investigator_names,

        "investigation_notes": investigation_notes,

        "investigation_result": investigation_result,

        "created_at": created_at.values,

        "resolved_at": resolved_at
    })

    return fraud_investigations_df



In [88]:

# ============================================================
# Generate Fraud Investigations
# ============================================================

fraud_investigations_df = generate_fraud_investigations(
    fraud_predictions_df
)

print(
    "Fraud Investigations:",
    len(fraud_investigations_df)
)

fraud_investigations_df.head()

Fraud cases selected for investigation: 2002
Fraud Investigations: 2002


,investigation_id,transaction_id,fraud_prediction_id,investigation_status,investigation_priority,investigator_name,investigation_notes,investigation_result,created_at,resolved_at
0,INV000001,TXN000011,FRD000011,Closed,High,Amara Guha,Medium-risk fraudulent transaction flagged for...,Confirmed Fraud,2024-02-18 02:23:40,2024-02-20 02:23:40
1,INV000002,TXN000023,FRD000023,Closed,High,Anthony Bal,Medium-risk fraudulent transaction flagged for...,False Positive,2024-10-21 15:54:57,2024-10-28 15:54:57
2,INV000003,TXN000031,FRD000031,Closed,High,Amara Arya,Medium-risk fraudulent transaction flagged for...,Confirmed Fraud,2026-06-02 11:51:11,2026-06-04 11:51:11
3,INV000004,TXN000067,FRD000067,Resolved,Critical,Chanakya Vala,High-risk fraudulent transaction requires imme...,False Positive,2024-09-22 12:29:05,2024-09-24 12:29:05
4,INV000005,TXN000129,FRD000129,In Progress,High,Damini Choudhary,Medium-risk fraudulent transaction flagged for...,None,2025-07-11 21:14:07,NaT


Row counts for all 16 tables

In [89]:
table_counts = {
    "Institution": len(institutions_df),
    "Customer": len(customers_df),
    "Account": len(accounts_df),
    "Card": len(cards_df),
    "Credit Profile": len(credit_profiles_df),
    "Loan": len(loans_df),
    "Merchant": len(merchants_df),
    "Payment Method": len(payment_methods_df),
    "Device": len(devices_df),
    "Location": len(locations_df),
    "Support Ticket": len(support_tickets_df),
    "Transactions": len(transactions_df),
    "Fraud Prediction": len(fraud_predictions_df),
    "Risk Score": len(risk_scores_df),
    "Churn Prediction": len(churn_predictions_df),
    "Fraud Investigation": len(fraud_investigations_df)
}

for table, count in table_counts.items():
    print(f"{table:<25} {count:,}")

Institution               5
Customer                  10,000
Account                   15,000
Card                      12,000
Credit Profile            10,000
Loan                      4,000
Merchant                  2,000
Payment Method            6
Device                    12,000
Location                  500
Support Ticket            5,000
Transactions              100,000
Fraud Prediction          100,000
Risk Score                10,000
Churn Prediction          10,000
Fraud Investigation       2,002


Check duplicate primary IDs

In [90]:
primary_keys = {
    "Institution": ("institution_id", institutions_df),
    "Customer": ("customer_id", customers_df),
    "Account": ("account_id", accounts_df),
    "Card": ("card_id", cards_df),
    "Credit Profile": ("credit_profile_id", credit_profiles_df),
    "Loan": ("loan_id", loans_df),
    "Merchant": ("merchant_id", merchants_df),
    "Payment Method": ("payment_method_id", payment_methods_df),
    "Device": ("device_id", devices_df),
    "Location": ("location_id", locations_df),
    "Support Ticket": ("ticket_id", support_tickets_df),
    "Transactions": ("transaction_id", transactions_df),
    "Fraud Prediction": ("fraud_prediction_id", fraud_predictions_df),
    "Risk Score": ("risk_score_id", risk_scores_df),
    "Churn Prediction": ("churn_prediction_id", churn_predictions_df),
    "Fraud Investigation": ("investigation_id", fraud_investigations_df)
}

for table, (column, df) in primary_keys.items():
    duplicates = df[column].duplicated().sum()
    print(f"{table:<25} duplicates: {duplicates}")

Institution               duplicates: 0
Customer                  duplicates: 0
Account                   duplicates: 0
Card                      duplicates: 0
Credit Profile            duplicates: 0
Loan                      duplicates: 0
Merchant                  duplicates: 0
Payment Method            duplicates: 0
Device                    duplicates: 0
Location                  duplicates: 0
Support Ticket            duplicates: 0
Transactions              duplicates: 0
Fraud Prediction          duplicates: 0
Risk Score                duplicates: 0
Churn Prediction          duplicates: 0
Fraud Investigation       duplicates: 0


## **Export all 16 CSVs**

In [91]:
import os

# Create data folder
os.makedirs("data", exist_ok=True)

# Export all 16 tables
institutions_df.to_csv("data/institution.csv", index=False)
customers_df.to_csv("data/customer.csv", index=False)
accounts_df.to_csv("data/account.csv", index=False)
cards_df.to_csv("data/card.csv", index=False)
credit_profiles_df.to_csv("data/credit_profile.csv", index=False)
loans_df.to_csv("data/loan.csv", index=False)
merchants_df.to_csv("data/merchant.csv", index=False)
payment_methods_df.to_csv("data/payment_method.csv", index=False)
devices_df.to_csv("data/device.csv", index=False)
locations_df.to_csv("data/location.csv", index=False)
support_tickets_df.to_csv("data/support_ticket.csv", index=False)
transactions_df.to_csv("data/transactions.csv", index=False)
fraud_predictions_df.to_csv("data/fraud_prediction.csv", index=False)
risk_scores_df.to_csv("data/risk_score.csv", index=False)
churn_predictions_df.to_csv("data/churn_prediction.csv", index=False)
fraud_investigations_df.to_csv("data/fraud_investigation.csv", index=False)

print("All 16 tables exported successfully.")

All 16 tables exported successfully.


In [92]:
import shutil

shutil.make_archive(
    "sufinex_data",
    "zip",
    "data"
)

print("Created sufinex_data.zip")

Created sufinex_data.zip


In [93]:
from google.colab import files

files.download("sufinex_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>